In [ ]:

!pip -q install pypdf faiss-cpu sentence-transformers gradio transformers torch hf_xet

Loading and Parsing the PDF: To extract the text from the PDF, I wrote a function called extract_text_from_pdf to parse the PDF report. The file is opened in binary mode and passed into PdfReader, and I looped through each page using extract_text(). To preserve ease of reading, I added a newline between pages. 

Chunking Strategy: I improved on the earlier approach by splitting the document into chunks at natural sentence boundaries using a regular expression. Each chunk was capped at around 400 characters  This meant that sentences stayed whole instead of being cut off in the middle, resolving the problem of losing context. I also filtered out very short pieces of text so that the chunks fed into the embedding model would always contain meaningful information. This strategy preserved ideas such as full findings, making it easier for the system to full  context.

Embeddings: I then loaded the all-MiniLM-L6-v2 model from SentenceTransformer because it was told to be a balance between speed and semantic accuracy. This this is suitable for lightweight application such as this one. I encoded each chunk into a vector embedding. I determined the vector dimension from the embedding shape .I then passed that into FAISS to make sure the index matched the model output.

Building the FAISS Index: For the similarity search, I went with IndexFlatL2 from FAISS , which calculates exact nearest-neighbor matches using L2 distance. I chose this method because it’s simple, and works well with sentence embeddings for relevance ranking. Before adding the embeddings, I converted them to float32 format since FAISS expects that data type.

Creating embeddings: WNext, I created embeddings for each chunk using the all-MiniLM-L6-v2 model. These embeddings transform text into numerical vectors that capture semantic meaning. To build the similarity search engine, I used FAISS with IndexFlatIP that usesinner product since this setup works well with normalized embeddings. I normalized the vectors using faiss.normalize_L2 before indexing them.This is to  ensure that retrieval was based on semantic closeness rather than just word patterns. This meant that the system could return the most relevant chunks when answering queries..

Setup of RAG pipeline: The retrieve_context function takes a user query, converts it into an embedding, normalizes it, and searches the FAISS index for the top k most relevant text chunks ( the default being 5). It then retrieves those chunks from the document and joins them together with this --- separators. It then returns  a readable block of context that best matches the query

Implementation of LLM:First, the code loads the FLAN-T5 base model from HuggingFace. It uses a tokenizer to break text into tokens that the model can understand.Themodel itself is designed to take in text and generate new text as output. These two are then combined into a pipeline for text-to-text generation. The pipeline is configured to generate answers with a limit on token length, and sampling parameters (temperature and top_p) are set to control the responses variance.The get_answer function is where the actual question answering happens. When the user asks a question, the function first calls retrieve_context to pull out the five most relevant text chunks from the PDF. If the retrieved context is too long (over 1500 characters), it is shortened to keep the input manageable. Next, the question and the retrieved context are formatted together into a prompt telling the model to “Answer this question based on the provided context.”The promp follows the recommended format from the FLAN API documentation.This prompt is then fed into the FLAN T5 pipeline, which generates a response.

In [11]:
#Import Libraries
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr
from transformers import AutoTokenizer, T5ForConditionalGeneration, pipeline
import torch
import re
from transformers import AutoTokenizer, T5ForConditionalGeneration
pdf_path = r"C:\Users\Al\Desktop\Questions_RND\Question3\COVID19_sitrep_MYS_w-46--47.pdf"

# Parsing the documment
def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file"""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

Creating text chunks
def text_chunking(text, chunk_size=400):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk + sentence) <= chunk_size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return [chunk for chunk in chunks if len(chunk) > 20]

doc_text = extract_text_from_pdf(r"C:\Users\Al\Desktop\Questions_RND\Question3\COVID19_sitrep_MYS_w-46--47.pdf")
chunks = text_chunking(doc_text)

#Create Embeddings & FAISS Index
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1]) # Get dimension of the embedding
faiss.normalize_L2(chunk_embeddings)
index.add(chunk_embeddings)

# Setup of the RAG Pipeline with the implemeentation of an open source LLM
def retrieve_context(query, k=5):
    query_embedding = embedder.encode([query])
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, k)
    
    # Return more context with chunk separation
    relevant_chunks = [chunks[i] for i in indices[0]]
    return "\n\n---\n\n".join(relevant_chunks)


# Using FLAN-T5 Base 

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Create pipeline for text-to-text generation(FLAN-T5 base)
llm_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3,
    top_p=0.9
)

print("Model loaded successfully!")

# Answer Function 
def get_answer(question):
    try:
        context = retrieve_context(question, k=5)
        
        if len(context) > 1500:  # Keepinfg context manageable
            context = context[:1500]
        
        # Formatting the prompt based on  FLAN-T5 frecommended API documentation
        prompt = f"Answer this question based on the provided context: {question}\n\nContext: {context}"
        
        # Generating response with FLAN T5
        response = llm_pipeline(
            prompt,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.3,
            top_p=0.9
        )
        
        # Extract the answer (FLAN-T5 returns clean text)
        if response and len(response) > 0:
            answer = response[0]['generated_text'].strip()
        else:
            answer = "Couldn't generate response based on context."
        
        response_text = f"Answer: {answer}"
        response_text += f"\n\nRelevant context preview:\n{context[:300]}..."
        
        return response_text
        
    except Exception as e:
        return f"Error processing question: {str(e)}\n\nContext found:\n{context[:500] if 'context' in locals() else 'No context retrieved'}..."

# Simple Gradio Interface
iface = gr.Interface(
    fn=get_answer,
    inputs=gr.Textbox(label="Ask about COVID-19 report"),
    outputs=gr.Textbox(label="Answer"),
    title="COVID-19 Report QA System",
    description="Ask specific questions about Malaysia's Covid-19 situation report",
    examples=[
        ["What was the total number of confirmed COVID-19 cases as of 27 November 2022?"],
        ["Which two states had the highest 14-day positivity rates, and what were the rates?"],
        ["What are the key findings in this report?"],
        ["How many new cases were reported?"]
    ]
)

print("Starting Gradio interface..")
iface.launch(share=True)

Loading model: google/flan-t5-base


Device set to use cpu


Model loaded successfully!
Starting Gradio interface..
* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://0b6e9b1e590665019d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
